In [11]:
# part 4 
# Shors algorithm. Shors gives us a much faster method of finding factors in polynomial time

# Install if needed:
# pip install qiskit qiskit-aer numpy pylatexenc
import numpy as np
from fractions import Fraction
from math import gcd

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile #added
from qiskit.circuit.library import QFTGate, UnitaryGate
from qiskit_aer import AerSimulator


def modular_multiplication_unitary(a: int, N: int, num_work_qubits: int) -> np.ndarray:
    """Return the unitary matrix for |y> -> |a*y mod N>.

    Basis states y >= N are left unchanged so the operation remains a
    permutation on the full 2**num_work_qubits-dimensional Hilbert space.
    This simple construction is useful for small educational examples.
    """
    dimension = 2 ** num_work_qubits
    matrix = np.zeros((dimension, dimension), dtype=complex)

    for y in range(dimension):
        if y < N:
            target = (a * y) % N
        else:
            target = y

        matrix[target, y] = 1

    return matrix


def build_order_finding_circuit(N: int, a: int, counting_qubits: int, work_qubits: int):
    counting = QuantumRegister(counting_qubits, "counting")
    work = QuantumRegister(work_qubits, "work")
    classical = ClassicalRegister(counting_qubits, "c")

    qc = QuantumCircuit(counting, work, classical)

    # Put the work register in |1>.
    qc.x(work[0])

    # Create a superposition over exponent values.
    qc.h(counting)

    # Apply controlled modular multiplications by a^(2^j) mod N.
    for j in range(counting_qubits):
        power = 2 ** j
        multiplier = pow(a, power, N)

        unitary = modular_multiplication_unitary(multiplier, N, work_qubits)
        gate = UnitaryGate(unitary, label=f"{a}^{power} mod {N}").control()

        qc.append(gate, [counting[j]] + list(work))

    # Apply inverse QFT to reveal the period information.
    qc.append(QFTGate(counting_qubits).inverse(), counting)

    # Measure only the counting register.
    qc.measure(counting, classical)

    return qc


def recover_order_from_counts(counts, Q: int, N: int, a: int):
    """Recover a valid order candidate from phase-estimation counts."""
    sorted_counts = sorted(counts.items(), key=lambda item: item[1], reverse=True)

    for bitstring, frequency in sorted_counts:
        y = int(bitstring, 2)

        if y == 0:
            continue

        fraction = Fraction(y, Q).limit_denominator(N)
        r_candidate = fraction.denominator

        if r_candidate > 0 and pow(a, r_candidate, N) == 1:
            return r_candidate, bitstring, frequency, fraction

    return None


N = 15
a = 4
counting_qubits = 4
work_qubits = 4
Q = 2 ** counting_qubits

qc = build_order_finding_circuit(N, a, counting_qubits, work_qubits)

simulator = AerSimulator()
qc_new = transpile(qc, simulator) # added? transpilation seems to fix it
job = simulator.run(qc_new, shots=2048)
result = job.result()
counts = result.get_counts()

print("Measurement counts:")
print(counts)

recovered = recover_order_from_counts(counts, Q, N, a)

if recovered is None:
    print("No valid order was recovered. Try increasing shots.")
else:
    r, bitstring, frequency, fraction = recovered

    print("Selected bitstring:", bitstring)
    print("Approximation y/Q:", fraction)
    print("Recovered order r:", r)

    x = pow(a, r // 2, N)

    factor_1 = gcd(x - 1, N)
    factor_2 = gcd(x + 1, N)

    print("Factors:", factor_1, factor_2)

Measurement counts:
{'1000': 1032, '0000': 1016}
Selected bitstring: 1000
Approximation y/Q: 1/2
Recovered order r: 2
Factors: 3 5


In [7]:
# Install if needed:
# pip install qiskit-ibm-runtime

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Load your saved IBM Quantum account.
# You can save your token once using:
# QiskitRuntimeService.save_account(channel="ibm_quantum", token="YOUR_IBM_QUANTUM_TOKEN")
service = QiskitRuntimeService()

backend = service.least_busy(
    operational=True,
    simulator=False,
    min_num_qubits=8
)

pm = generate_preset_pass_manager(
    backend=backend,
    optimization_level=2
)

isa_circuit = pm.run(qc)

sampler = Sampler(mode=backend)
job = sampler.run([isa_circuit], shots=1024)

print("Backend:", backend.name)
print("Job ID:", job.job_id())

result = job.result()
counts = result[0].data.c.get_counts()

print(counts)

Backend: ibm_kingston
Job ID: d9bk3fm6hjac73ffqpcg
{'1000': 302, '1101': 11, '0000': 256, '0001': 18, '1010': 21, '0100': 122, '1100': 200, '0101': 13, '0110': 15, '1110': 26, '1111': 2, '0011': 3, '0010': 19, '1001': 12, '1011': 2, '0111': 2}


In [8]:
recovered = recover_order_from_counts(counts, Q, N, a)

if recovered is None:
    print("No valid order was recovered. Try increasing shots.")
else:
    r, bitstring, frequency, fraction = recovered

    print("Selected bitstring:", bitstring)
    print("Approximation y/Q:", fraction)
    print("Recovered order r:", r)

    x = pow(a, r // 2, N)

    factor_1 = gcd(x - 1, N)
    factor_2 = gcd(x + 1, N)

    print("Factors:", factor_1, factor_2)

Selected bitstring: 1100
Approximation y/Q: 3/4
Recovered order r: 4
Factors: 3 5
